In [ ]:
from ultralytics import YOLO
from sahi.slicing import slice_image
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
import torchvision.ops as ops

In [ ]:
# -- Configuration -- edit these before running
MODEL_PATH    = "runs/cfu_detector/weights/best.pt"
IMAGE_DIR     = "data/images_resized"  # full-resolution images to run inference on
CONF          = 0.25
IOU           = 0.45
GRID          = 10        # must match training_prep.ipynb
OVERLAP_RATIO = 0.1       # must match training_prep.ipynb

CLASS_NAMES = ['GM', 'BFU-E', 'E', 'GEMM']
COLORS      = ['#e6194b', '#3cb44b', '#4363d8', '#f58231']

In [ ]:
model = YOLO(MODEL_PATH)
print(f"Loaded model from {MODEL_PATH}")

In [ ]:
def predict_full_image(image_path, conf=CONF, iou=IOU, grid=GRID, overlap_ratio=OVERLAP_RATIO):
    """
    Tiles a full-resolution image using the same parameters as training,
    runs inference on each tile, remaps boxes to original image coordinates,
    and applies NMS across tiles to remove boundary duplicates.

    Returns:
        boxes  (N, 4): xyxy in original image coordinates
        scores (N,):   confidence scores
        labels (N,):   class indices
    """
    img = Image.open(image_path)
    W, H = img.size
    tile_w = W // grid
    tile_h = H // grid

    slice_result = slice_image(
        image=str(image_path),
        slice_height=tile_h,
        slice_width=tile_w,
        overlap_height_ratio=overlap_ratio,
        overlap_width_ratio=overlap_ratio,
        verbose=False,
    )

    all_boxes, all_scores, all_labels = [], [], []

    for sliced in slice_result.sliced_image_list:
        tx1 = sliced.starting_pixel[0]
        ty1 = sliced.starting_pixel[1]
        tile_img = Image.fromarray(sliced.image)

        results = model.predict(tile_img, conf=conf, iou=iou, verbose=False)
        result  = results[0]

        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            all_boxes.append([x1 + tx1, y1 + ty1, x2 + tx1, y2 + ty1])
            all_scores.append(float(box.conf[0]))
            all_labels.append(int(box.cls[0]))

    if not all_boxes:
        return np.empty((0, 4)), np.array([]), np.array([])

    boxes_t  = torch.tensor(all_boxes,  dtype=torch.float32)
    scores_t = torch.tensor(all_scores, dtype=torch.float32)
    labels_t = torch.tensor(all_labels, dtype=torch.int32)

    # Per-class NMS to avoid suppressing true detections of different classes
    keep_indices = []
    for cls_idx in labels_t.unique():
        mask = labels_t == cls_idx
        keep = ops.nms(boxes_t[mask], scores_t[mask], iou)
        keep_indices.append(mask.nonzero(as_tuple=True)[0][keep])

    keep = torch.cat(keep_indices)
    return boxes_t[keep].numpy(), scores_t[keep].numpy(), labels_t[keep].numpy()

In [ ]:
def plot_detections(image_path, boxes, scores, labels, figsize=(14, 14)):
    img = np.array(Image.open(image_path))
    fig, ax = plt.subplots(1, figsize=figsize)
    ax.imshow(img)

    for (x1, y1, x2, y2), score, cls in zip(boxes, scores, labels):
        color = COLORS[cls % len(COLORS)]
        rect  = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=1, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"{CLASS_NAMES[cls]} {score:.2f}",
                color=color, fontsize=6, fontweight='bold')

    counts = {name: int((labels == i).sum()) for i, name in enumerate(CLASS_NAMES)}
    ax.set_title(f"{Path(image_path).name}  |  {counts}")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Run inference on all images and collect counts
supported   = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
image_paths = sorted(p for p in Path(IMAGE_DIR).iterdir() if p.suffix.lower() in supported)
print(f"Found {len(image_paths)} images")

rows = []
all_results = {}

for img_path in image_paths:
    boxes, scores, labels = predict_full_image(img_path)
    all_results[img_path] = (boxes, scores, labels)
    counts = {name: int((labels == i).sum()) for i, name in enumerate(CLASS_NAMES)}
    rows.append({'image': img_path.name, **counts})
    print(f"{img_path.name}: {counts}")

df = pd.DataFrame(rows)
df['total'] = df[CLASS_NAMES].sum(axis=1)

In [ ]:
# Summary table
display(df)
print("\nTotals:")
print(df[CLASS_NAMES + ['total']].sum().to_string())

In [ ]:
# Visualise detections on each image
for img_path, (boxes, scores, labels) in all_results.items():
    plot_detections(img_path, boxes, scores, labels)

In [ ]:
# Save counts to CSV
out_csv = Path("runs/cfu_detector/inference_counts.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)
print(f"Saved to {out_csv}")